##### !pip install pandas

In [1]:
!pip install matplotlib

In [2]:
!pip install seaborn

In [3]:
!pip install scikit-learn

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn import svm
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import RandomizedSearchCV
from collections import Counter

In [5]:
import pandas as pd

pd.set_option('display.max_colwidth', None)


In [6]:
dis_sym_data = pd.read_csv(r"Original_Dataset.csv")

In [7]:
dis_sym_data

,Disease,Symptom_1,Symptom_2,Symptom_3,Symptom_4,Symptom_5,Symptom_6,Symptom_7,Symptom_8,Symptom_9,Symptom_10,Symptom_11,Symptom_12,Symptom_13,Symptom_14,Symptom_15,Symptom_16,Symptom_17
0,Fungal infection,itching,skin_rash,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Fungal infection,itching,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Fungal infection,skin_rash,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Fungal infection,itching,nodal_skin_eruptions,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Fungal infection,itching,skin_rash,dischromic _patches,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4532,(vertigo) Paroymsal Positional Vertigo,vomiting,headache,nausea,spinning_movements,loss_of_balance,unsteadiness,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4533,Acne,skin_rash,pus_filled_pimples,blackheads,scurring,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4534,Urinary tract infection,burning_micturition,bladder_discomfort,foul_smell_of urine,continuous_feel_of_urine,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4535,Psoriasis,skin_rash,joint_pain,skin_peeling,silver_like_dusting,small_dents_in_nails,inflammatory_nails,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
columns_to_check = []
for col in dis_sym_data.columns:
    if col != 'Disease':
        columns_to_check.append(col)

In [9]:
symptoms = dis_sym_data.iloc[:, 1:].values.flatten()
symptoms = list(set(symptoms))

In [10]:
for symptom in symptoms:
    dis_sym_data[symptom] = dis_sym_data.iloc[:, 1:].apply(lambda row: int(symptom in row.values), axis=1)

dis_sym_data_v1 = dis_sym_data.drop(columns=columns_to_check)

In [11]:
dis_sym_data_v1 = dis_sym_data_v1.loc[:, dis_sym_data_v1.columns.notna()]

In [12]:
dis_sym_data_v1.shape

(4537, 133)

In [13]:
dis_sym_data_v1.columns = dis_sym_data_v1.columns.str.strip()

In [14]:
dis_sym_data_v1.columns

Index(['Disease', 'dark_urine', 'weight_loss', 'blackheads', 'coma',
       'yellowing_of_eyes', 'movement_stiffness', 'rusty_sputum',
       'swollen_extremeties', 'swelled_lymph_nodes',
       ...
       'weakness_in_limbs', 'dehydration', 'excessive_hunger', 'blister',
       'bruising', 'small_dents_in_nails', 'polyuria', 'back_pain',
       'congestion', 'constipation'],
      dtype='object', length=133)

In [15]:
dis_sym_data_v1

,Disease,dark_urine,weight_loss,blackheads,coma,yellowing_of_eyes,movement_stiffness,rusty_sputum,swollen_extremeties,swelled_lymph_nodes,...,weakness_in_limbs,dehydration,excessive_hunger,blister,bruising,small_dents_in_nails,polyuria,back_pain,congestion,constipation
0,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4532,(vertigo) Paroymsal Positional Vertigo,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4533,Acne,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4534,Urinary tract infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4535,Psoriasis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [16]:
# Select columns that start with "i"
columns_starting_with_i = [col for col in dis_sym_data_v1.columns if col.startswith('itching')]

# Display the rows with only these columns
print(dis_sym_data_v1[columns_starting_with_i])


      itching
0           1
1           1
2           0
3           1
4           1
...       ...
4532        0
4533        0
4534        0
4535        0
4536        0

[4537 rows x 1 columns]


In [17]:
dis_sym_data_v1[columns_starting_with_i]

,itching
0,1
1,1
2,0
3,1
4,1
...,...
4532,0
4533,0
4534,0
4535,0


In [18]:
dis_sym_data_v1

,Disease,dark_urine,weight_loss,blackheads,coma,yellowing_of_eyes,movement_stiffness,rusty_sputum,swollen_extremeties,swelled_lymph_nodes,...,weakness_in_limbs,dehydration,excessive_hunger,blister,bruising,small_dents_in_nails,polyuria,back_pain,congestion,constipation
0,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Fungal infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4532,(vertigo) Paroymsal Positional Vertigo,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4533,Acne,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4534,Urinary tract infection,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4535,Psoriasis,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0


In [19]:
dis_sym_data_v1.to_csv("Processed_Dataset1.csv")

In [20]:
symptom = list(dis_sym_data_v1)
symptom

['Disease',
 'dark_urine',
 'weight_loss',
 'blackheads',
 'coma',
 'yellowing_of_eyes',
 'movement_stiffness',
 'rusty_sputum',
 'swollen_extremeties',
 'swelled_lymph_nodes',
 'swelling_of_stomach',
 'phlegm',
 'mild_fever',
 'sunken_eyes',
 'nausea',
 'watering_from_eyes',
 'stomach_pain',
 'pain_in_anal_region',
 'abnormal_menstruation',
 'mucoid_sputum',
 'swollen_legs',
 'brittle_nails',
 'silver_like_dusting',
 'enlarged_thyroid',
 'depression',
 'inflammatory_nails',
 'distention_of_abdomen',
 'bladder_discomfort',
 'receiving_blood_transfusion',
 'puffy_face_and_eyes',
 'visual_disturbances',
 'abdominal_pain',
 'headache',
 'pain_behind_the_eyes',
 'acute_liver_failure',
 'weakness_of_one_body_side',
 'cough',
 'unsteadiness',
 'skin_peeling',
 'fluid_overload',
 'fatigue',
 'lethargy',
 'diarrhoea',
 'irregular_sugar_level',
 'continuous_feel_of_urine',
 'cramps',
 'yellow_urine',
 'sinus_pressure',
 'vomiting',
 'scurring',
 'anxiety',
 'increased_appetite',
 'mood_swings',

In [21]:
var_mod = ['Disease']
le = LabelEncoder()
for i in var_mod:
    dis_sym_data_v1[i] = le.fit_transform(dis_sym_data_v1[i])

In [22]:
# Save the encoder using pickle
import pickle
with open('label_encoder.pkl', 'wb') as file:
    pickle.dump(le, file)

In [23]:
X = dis_sym_data_v1.drop(columns="Disease")
y = dis_sym_data_v1['Disease']

In [24]:
X.shape

(4537, 132)

In [25]:
for i in X.columns:
    print(i)

dark_urine
weight_loss
blackheads
coma
yellowing_of_eyes
movement_stiffness
rusty_sputum
swollen_extremeties
swelled_lymph_nodes
swelling_of_stomach
phlegm
mild_fever
sunken_eyes
nausea
watering_from_eyes
stomach_pain
pain_in_anal_region
abnormal_menstruation
mucoid_sputum
swollen_legs
brittle_nails
silver_like_dusting
enlarged_thyroid
depression
inflammatory_nails
distention_of_abdomen
bladder_discomfort
receiving_blood_transfusion
puffy_face_and_eyes
visual_disturbances
abdominal_pain
headache
pain_behind_the_eyes
acute_liver_failure
weakness_of_one_body_side
cough
unsteadiness
skin_peeling
fluid_overload
fatigue
lethargy
diarrhoea
irregular_sugar_level
continuous_feel_of_urine
cramps
yellow_urine
sinus_pressure
vomiting
scurring
anxiety
increased_appetite
mood_swings
family_history
extra_marital_contacts
foul_smell_of urine
stiff_neck
slurred_speech
red_sore_around_nose
loss_of_smell
breathlessness
drying_and_tingling_lips
muscle_weakness
loss_of_balance
palpitations
high_fever
rest

In [26]:
y

0       15
1       15
2       15
3       15
4       15
        ..
4532     0
4533     2
4534    38
4535    35
4536    27
Name: Disease, Length: 4537, dtype: int64

In [27]:
import joblib
def class_algo(model,independent,dependent):
    model.fit(independent,dependent)
    pred = model.predict(independent)
    accuracy = metrics.accuracy_score(pred,dependent) 
    print(model_name,'Accuracy : %s' % '{0:.3%}'.format(accuracy))
    model_filename = f"{model.__class__.__name__}_model.pkl"
    joblib.dump(model, model_filename)  # Save the model to a file
    print(f"Model saved as {model_filename}\n")


In [28]:
algorithms = {'Logistic Regression': 
              {"model": LogisticRegression()},
              
              'Decision Tree': 
              {"model": tree.DecisionTreeClassifier()},
              
              'Random Forest': 
              {"model": RandomForestClassifier()},
              
              'SVM':
              {"model": svm.SVC(probability=True)},
              
              'NaiveBayes' :
              {"model": GaussianNB()},
              
              'K-Nearest Neighbors' :
              {"model": KNeighborsClassifier()},
             }

In [29]:
for model_name, values in algorithms.items():
    class_algo(values["model"],X,y)

Logistic Regression Accuracy : 99.978%
Model saved as LogisticRegression_model.pkl

Decision Tree Accuracy : 99.978%
Model saved as DecisionTreeClassifier_model.pkl

Random Forest Accuracy : 99.978%
Model saved as RandomForestClassifier_model.pkl

SVM Accuracy : 99.868%
Model saved as SVC_model.pkl

NaiveBayes Accuracy : 99.956%
Model saved as GaussianNB_model.pkl

K-Nearest Neighbors Accuracy : 99.912%
Model saved as KNeighborsClassifier_model.pkl



In [30]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import joblib

# Load the dataset
pd.set_option('display.max_colwidth', None)
dis_sym_data = pd.read_csv(r"Original_Dataset.csv")

# Ensure column names are strings to avoid errors
dis_sym_data.columns = dis_sym_data.columns.astype(str)

# Generate binary features for symptoms
columns_to_check = [col for col in dis_sym_data.columns if col != 'Disease']

# Flatten symptoms, deduplicate, and create binary features
symptoms = dis_sym_data.iloc[:, 1:].values.flatten()
symptoms = list(set(symptoms[~pd.isnull(symptoms)]))  # Remove NaN values

for symptom in symptoms:
    dis_sym_data[symptom] = dis_sym_data.iloc[:, 1:].apply(
        lambda row: int(symptom in row.values), axis=1
    )

# Drop original symptom columns to keep only binary features
dis_sym_data_v1 = dis_sym_data.drop(columns=columns_to_check)

# Ensure no NaN column names exist
dis_sym_data_v1 = dis_sym_data_v1.loc[:, dis_sym_data_v1.columns.notna()]

# Strip whitespace from column names
dis_sym_data_v1.columns = dis_sym_data_v1.columns.str.strip()

# Encode the target variable
var_mod = ['Disease']
le = LabelEncoder()
for col in var_mod:
    dis_sym_data_v1[col] = le.fit_transform(dis_sym_data_v1[col])

# Split data into features and target
X = dis_sym_data_v1.drop(columns="Disease")
y = dis_sym_data_v1['Disease']

# Define a function to train models and save them
def class_algo(model, independent, dependent, model_name):
    model.fit(independent, dependent)
    pred = model.predict(independent)
    accuracy = metrics.accuracy_score(pred, dependent)
    print(f"{model_name} Accuracy: {accuracy:.3%}")
    model_filename = f"{model.__class__.__name__}_model.pkl"
    joblib.dump(model, model_filename)  # Save the model to a file
    print(f"Model saved as {model_filename}\n")

# Define algorithms
algorithms = {
    'Logistic Regression': {"model": LogisticRegression()},
    'Decision Tree': {"model": tree.DecisionTreeClassifier()},
    'Random Forest': {"model": RandomForestClassifier()},
    'SVM': {"model": svm.SVC(probability=True)},
    'Naive Bayes': {"model": GaussianNB()},
    'K-Nearest Neighbors': {"model": KNeighborsClassifier()},
}

# Train and save models
for model_name, values in algorithms.items():
    class_algo(values["model"], X, y, model_name)


Logistic Regression Accuracy: 99.978%
Model saved as LogisticRegression_model.pkl

Decision Tree Accuracy: 99.978%
Model saved as DecisionTreeClassifier_model.pkl

Random Forest Accuracy: 99.978%
Model saved as RandomForestClassifier_model.pkl

SVM Accuracy: 99.868%
Model saved as SVC_model.pkl

Naive Bayes Accuracy: 99.956%
Model saved as GaussianNB_model.pkl

K-Nearest Neighbors Accuracy: 99.912%
Model saved as KNeighborsClassifier_model.pkl



In [31]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn import metrics
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.ensemble import RandomForestClassifier
from sklearn import svm
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
import joblib

# Load the dataset
pd.set_option('display.max_colwidth', None)
dis_sym_data = pd.read_csv(r"Original_Dataset.csv")

# Ensure column names are strings to avoid errors
dis_sym_data.columns = dis_sym_data.columns.astype(str)

# Generate binary features for symptoms
columns_to_check = [col for col in dis_sym_data.columns if col != 'Disease']

# Flatten symptoms, deduplicate, and create binary features
symptoms = dis_sym_data.iloc[:, 1:].values.flatten()
symptoms = list(set(symptoms[~pd.isnull(symptoms)]))  # Remove NaN values

for symptom in symptoms:
    dis_sym_data[symptom] = dis_sym_data.iloc[:, 1:].apply(
        lambda row: int(symptom in row.values), axis=1
    )

# Drop original symptom columns to keep only binary features
dis_sym_data_v1 = dis_sym_data.drop(columns=columns_to_check)

# Ensure no NaN column names exist
dis_sym_data_v1 = dis_sym_data_v1.loc[:, dis_sym_data_v1.columns.notna()]

# Strip whitespace from column names
dis_sym_data_v1.columns = dis_sym_data_v1.columns.str.strip()

# Encode the target variable
var_mod = ['Disease']
le = LabelEncoder()
for col in var_mod:
    dis_sym_data_v1[col] = le.fit_transform(dis_sym_data_v1[col])

# Save the LabelEncoder for later use
joblib.dump(le, "label_encoder.pkl")
print("LabelEncoder saved as 'label_encoder.pkl'")

# Split data into features and target
X = dis_sym_data_v1.drop(columns="Disease")
y = dis_sym_data_v1['Disease']

# Define a function to train models
def train_and_save_models(models, X, y):
    trained_algorithms = {}

    for model_name, model_info in models.items():
        model = model_info["model"]
        model.fit(X, y)
        pred = model.predict(X)
        accuracy = metrics.accuracy_score(pred, y)
        print(f"{model_name} Accuracy: {accuracy:.3%}")
        trained_algorithms[model_name] = {"model": model, "accuracy": accuracy}

    # Save all trained models to a single file
    joblib.dump(trained_algorithms, "trained_algorithms.pkl")
    print("All trained algorithms saved as 'trained_algorithms.pkl'")

# Define algorithms
algorithms = {
    'Logistic Regression': {"model": LogisticRegression(max_iter=1000)},
    'Decision Tree': {"model": tree.DecisionTreeClassifier()},
    'Random Forest': {"model": RandomForestClassifier()},
    'SVM': {"model": svm.SVC(probability=True)},
    'Naive Bayes': {"model": GaussianNB()},
    'K-Nearest Neighbors': {"model": KNeighborsClassifier()},
}

# Train and save all models
train_and_save_models(algorithms, X, y)


LabelEncoder saved as 'label_encoder.pkl'
Logistic Regression Accuracy: 99.978%
Decision Tree Accuracy: 99.978%
Random Forest Accuracy: 99.978%
SVM Accuracy: 99.868%
Naive Bayes Accuracy: 99.956%
K-Nearest Neighbors Accuracy: 99.912%
All trained algorithms saved as 'trained_algorithms.pkl'


In [32]:
doc_data = pd.read_csv(r"Doctor_Versus_Disease.csv",encoding='latin1', names=['Disease','Specialist'])

In [33]:
doc_data

,Disease,Specialist
0,Drug Reaction,Allergist
1,Allergy,Allergist
2,Hypertension,Cardiologist
3,Heart attack,Cardiologist
4,Psoriasis,Dermatologist
5,Chicken pox,Dermatologist
6,Acne,Dermatologist
7,Impetigo,Dermatologist
8,Fungal infection,Dermatologist
9,Hypothyroidism,Endocrinologist


In [34]:
doc_data1= pd.read_csv(r"Doctor_Versus_Disease.csv",encoding='latin1', names=['Disease'])

In [35]:
doc_data1

,Disease
Drug Reaction,Allergist
Allergy,Allergist
Hypertension,Cardiologist
Heart attack,Cardiologist
Psoriasis,Dermatologist
Chicken pox,Dermatologist
Acne,Dermatologist
Impetigo,Dermatologist
Fungal infection,Dermatologist
Hypothyroidism,Endocrinologist


In [36]:
import numpy as np
doc_data['Specialist'] = np.where((doc_data['Disease'] == 'Tuberculosis'),'Pulmonologist', doc_data['Specialist'])

In [37]:
des_data = pd.read_csv(r"Disease_Description.csv")

In [38]:
des_data

,Disease,Description
0,Drug Reaction,An adverse drug reaction (ADR) is an injury caused by taking medication. ADRs may occur following a single dose or prolonged administration of a drug or result from the combination of two or more drugs.
1,Malaria,An infectious disease caused by protozoan parasites from the Plasmodium family that can be transmitted by the bite of the Anopheles mosquito or by a contaminated needle or transfusion. Falciparum malaria is the most deadly type.
2,Allergy,"An allergy is an immune system response to a foreign substance that's not typically harmful to your body.They can include certain foods, pollen, or pet dander. Your immune system's job is to keep you healthy by fighting harmful pathogens."
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroid or low thyroid, is a disorder of the endocrine system in which the thyroid gland does not produce enough thyroid hormone."
4,Psoriasis,"Psoriasis is a common skin disorder that forms thick, red, bumpy patches covered with silvery scales. They can pop up anywhere, but most appear on the scalp, elbows, knees, and lower back. Psoriasis can't be passed from person to person. It does sometimes happen in members of the same family."
5,GERD,"Gastroesophageal reflux disease, or GERD, is a digestive disorder that affects the lower esophageal sphincter (LES), the ring of muscle between the esophagus and stomach. Many people, including pregnant women, suffer from heartburn or acid indigestion caused by GERD."
6,Chronic cholestasis,"Chronic cholestatic diseases, whether occurring in infancy, childhood or adulthood, are characterized by defective bile acid transport from the liver to the intestine, which is caused by primary damage to the biliary epithelium in most cases"
7,hepatitis A,Hepatitis A is a highly contagious liver infection caused by the hepatitis A virus. The virus is one of several types of hepatitis viruses that cause inflammation and affect your liver's ability to function.
8,Osteoarthristis,"Osteoarthritis is the most common form of arthritis, affecting millions of people worldwide. It occurs when the protective cartilage that cushions the ends of your bones wears down over time."
9,(vertigo) Paroymsal Positional Vertigo,Benign paroxysmal positional vertigo (BPPV) is one of the most common causes of vertigo — the sudden sensation that you're spinning or that the inside of your head is spinning. Benign paroxysmal positional vertigo causes brief episodes of mild to intense dizziness.


In [39]:
des_data["Disease"]

0                               Drug Reaction
1                                     Malaria
2                                     Allergy
3                              Hypothyroidism
4                                   Psoriasis
5                                        GERD
6                         Chronic cholestasis
7                                 hepatitis A
8                             Osteoarthristis
9     (vertigo) Paroymsal  Positional Vertigo
10                               Hypoglycemia
11                                       Acne
12                                   Diabetes
13                                   Impetigo
14                               Hypertension
15                        Peptic ulcer diseae
16               Dimorphic hemorrhoids(piles)
17                                Common Cold
18                                Chicken pox
19                       Cervical spondylosis
20                            Hyperthyroidism
21                    Urinary trac

In [40]:
### from collections import Counter
test_col = []
for col in dis_sym_data_v1.columns:
    if col != 'Disease':
        test_col.append(col)
print(test_col)

test_data = {}
symptoms = []
predicted = []
from collections import Counter
import pandas as pd
import joblib

# Load trained models and label encoder
trained_algorithms = joblib.load("trained_algorithms.pkl")
le = joblib.load("label_encoder.pkl")

def test_input():
    symptoms = []
    predicted = []
    
    num_inputs = int(input("Enter the number of symptoms you have: "))
    for i in range(num_inputs):
        user_input = input(f"Enter Symptoms #{i+1}: ").strip().replace(" ", "_").lower()
        symptoms.append(user_input)

    print("\nSymptoms you have:", symptoms)

    # Create test data dictionary
    test_data = {col: 1 if col in symptoms else 0 for col in X.columns}

    # Convert to DataFrame and align columns
    test_df = pd.DataFrame([test_data])
    test_df = test_df.reindex(columns=X.columns, fill_value=0)  # Ensure same feature set

    print("\nPredicting Disease based on 6 ML algorithms...\n")

    for model_name, values in trained_algorithms.items():
        model = values["model"]
        predict_disease = model.predict(test_df)
        predict_disease = le.inverse_transform(predict_disease)
        predicted.extend(predict_disease)

    # Calculate probabilities
    disease_counts = Counter(predicted)
    percentage_per_disease = {disease: (count / 6) * 100 for disease, count in disease_counts.items()}
    
    # Convert to DataFrame
    result_df = pd.DataFrame({
        "Disease": list(percentage_per_disease.keys()),
        "Chances": list(percentage_per_disease.values())
    })

    print(result_df)

# Run the test input function
test_input()


['dark_urine', 'weight_loss', 'blackheads', 'coma', 'yellowing_of_eyes', 'movement_stiffness', 'rusty_sputum', 'swollen_extremeties', 'swelled_lymph_nodes', 'swelling_of_stomach', 'phlegm', 'mild_fever', 'sunken_eyes', 'nausea', 'watering_from_eyes', 'stomach_pain', 'pain_in_anal_region', 'abnormal_menstruation', 'mucoid_sputum', 'swollen_legs', 'brittle_nails', 'silver_like_dusting', 'enlarged_thyroid', 'depression', 'inflammatory_nails', 'distention_of_abdomen', 'bladder_discomfort', 'receiving_blood_transfusion', 'puffy_face_and_eyes', 'visual_disturbances', 'abdominal_pain', 'headache', 'pain_behind_the_eyes', 'acute_liver_failure', 'weakness_of_one_body_side', 'cough', 'unsteadiness', 'skin_peeling', 'fluid_overload', 'fatigue', 'lethargy', 'diarrhoea', 'irregular_sugar_level', 'continuous_feel_of_urine', 'cramps', 'yellow_urine', 'sinus_pressure', 'vomiting', 'scurring', 'anxiety', 'increased_appetite', 'mood_swings', 'family_history', 'extra_marital_contacts', 'foul_smell_of uri

Enter the number of symptoms you have:  3
Enter Symptoms #1:  blister
Enter Symptoms #2:  malaise
Enter Symptoms #3:  chest_pain



Symptoms you have: ['blister', 'malaise', 'chest_pain']

Predicting Disease based on 6 ML algorithms...

        Disease    Chances
0  Heart attack  50.000000
1   Chicken pox  16.666667
2      Impetigo  16.666667
3  Tuberculosis  16.666667


In [41]:
test_input()

Enter the number of symptoms you have:  3
Enter Symptoms #1:  cold
Enter Symptoms #2:  cough
Enter Symptoms #3:  chills



Symptoms you have: ['cold', 'cough', 'chills']

Predicting Disease based on 6 ML algorithms...

        Disease    Chances
0       Malaria  50.000000
1        Dengue  33.333333
2  Tuberculosis  16.666667


In [42]:
pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [43]:
import gradio as gr
import pandas as pd
from collections import Counter

# Assume `dis_sym_data_v1`, `doc_data`, `des_data`, `algorithms`, and `le` are preloaded or defined elsewhere

# Column names excluding 'Disease'
test_col = [col for col in dis_sym_data_v1.columns if col != 'Disease']

# Store chatbot state globally
symptoms = []
num_symptoms = 0
current_step = 0

def reset_state():
    global symptoms, num_symptoms, current_step
    symptoms = []
    num_symptoms = 0
    current_step = 0

# Function to predict disease once all symptoms are gathered
def predict_disease_chatbot():
    test_data = {}
    predicted = []

    # Prepare test data for prediction
    for column in test_col:
        test_data[column] = 1 if column in symptoms else 0
    test_df = pd.DataFrame(test_data, index=[0])

    # Predict using multiple algorithms and collect predictions
    for model_name, values in algorithms.items():
        predict_disease = values["model"].predict(test_df)
        predict_disease = le.inverse_transform(predict_disease)
        predicted.extend(predict_disease)

    # Calculate the chances of each predicted disease
    disease_counts = Counter(predicted)
    percentage_per_disease = {disease: (count / 6) * 100 for disease, count in disease_counts.items()}

    # Convert to DataFrame
    result_df = pd.DataFrame({"Disease": list(percentage_per_disease.keys()),
                              "Chances": list(percentage_per_disease.values())})

    # Merge additional information (doctor and description)
    result_df = result_df.merge(doc_data, on='Disease', how='left')
    result_df = result_df.merge(des_data, on='Disease', how='left')

    # Format the response
    response = "Here are the predictions based on your symptoms:\n"
    for index, row in result_df.iterrows():
        doctor_info = row.get('Doctor', 'Doctor info not available')
        description_info = row.get('Description', 'Description not available')
        response += f"Disease: {row['Disease']}\nChances: {row['Chances']:.2f}%\nDoctor: {doctor_info}\nDescription: {description_info}\n\n"
    
    reset_state()  # Reset state after prediction
    return response

# Function to handle user inputs step by step, while maintaining the chat history
def chatbot_interface(user_input, chat_history=[]):
    global symptoms, num_symptoms, current_step

    if current_step == 0:
        # Step 1: Ask how many symptoms the user has
        try:
            num_symptoms = int(user_input)
            current_step = 1
            response = f"Okay, you have {num_symptoms} symptoms. Please enter symptom #1."
        except ValueError:
            response = "Please enter a valid number of symptoms."

    elif current_step <= num_symptoms:
        # Step 2: Collect symptoms one by one
        symptoms.append(user_input)
        if current_step < num_symptoms:
            current_step += 1
            response = f"Please enter symptom #{current_step}."
        else:
            # Step 3: Make the prediction
            response = predict_disease_chatbot()

    # Append the user input and chatbot response to the chat history
    chat_history.append((f"User: {user_input}", f"Chatbot: {response}"))
    
    return chat_history, chat_history  # Return the updated chat history and state

# Gradio chatbot interface with a submit button
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=[gr.Textbox(label="Enter your message", placeholder="Please Enter How many symptoms you have....", lines=1), gr.State([])],  # Textbox input and chat history state
    outputs=[gr.Chatbot(), gr.State([])],  # Output chatbot conversation and chat history state
    title="Disease Prediction Chatbot",
    description="Chatbot to predict possible diseases based on symptoms. First, enter the number of symptoms, and then enter each symptom one by one.",
    allow_flagging="never"
)

# Launch the chatbot interface
iface.launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [44]:
import gradio as gr
import pandas as pd
from collections import Counter
import joblib  # Import joblib for loading the saved models

# Assume the following files for the models:
model_files = {
    'Logistic Regression': 'LogisticRegression_model.pkl',
    'Decision Tree': 'DecisionTreeClassifier_model.pkl',
    'Random Forest': 'RandomForestClassifier_model.pkl',
    'SVM': 'SVC_model.pkl',
    'NaiveBayes': 'GaussianNB_model.pkl',
    'K-Nearest Neighbors': 'KNeighborsClassifier_model.pkl'
}

# Load the models from .pkl files
algorithms = {}
for model_name, model_file in model_files.items():
    algorithms[model_name] = {"model": joblib.load(model_file)}

# Assuming `dis_sym_data_v1`, `doc_data`, `des_data`, and `le` are preloaded or defined elsewhere

# Column names excluding 'Disease'
test_col = [col for col in dis_sym_data_v1.columns if col != 'Disease']

# Store chatbot state globally
symptoms = []
num_symptoms = 0
current_step = 0

def reset_state():
    global symptoms, num_symptoms, current_step
    symptoms = []
    num_symptoms = 0
    current_step = 0

# Function to predict disease once all symptoms are gathered
def predict_disease_chatbot():
    test_data = {}
    predicted = []

    # Prepare test data for prediction
    for column in test_col:
        test_data[column] = 1 if column in symptoms else 0
    test_df = pd.DataFrame(test_data, index=[0])
    print(test_df)
    # Predict using multiple algorithms and collect predictions
    for model_name, values in algorithms.items():
        predict_disease = values["model"].predict(test_df)
        predict_disease = le.inverse_transform(predict_disease)
        predicted.extend(predict_disease)
    
    # Calculate the chances of each predicted disease
    disease_counts = Counter(predicted)
    percentage_per_disease = {disease: (count / 6) * 100 for disease, count in disease_counts.items()}

    # Convert to DataFrame
    result_df = pd.DataFrame({"Disease": list(percentage_per_disease.keys()),
                              "Chances": list(percentage_per_disease.values())})

    # Merge additional information (doctor and description)
    result_df = result_df.merge(doc_data, on='Disease', how='left')
    result_df = result_df.merge(des_data, on='Disease', how='left')

    # Format the response
    response = "Here are the predictions based on your symptoms:\n"
    for index, row in result_df.iterrows():
        doctor_info = row.get('Doctor', 'Doctor info not available')
        description_info = row.get('Description', 'Description not available')
        response += f"Disease: {row['Disease']}\nChances: {row['Chances']:.2f}%\nDoctor: {doctor_info}\nDescription: {description_info}\n\n"
    
    reset_state()  # Reset state after prediction
    return response

# Function to handle user inputs step by step, while maintaining the chat history
def chatbot_interface(user_input, chat_history=[]):
    global symptoms, num_symptoms, current_step

    if current_step == 0:
        # Step 1: Ask how many symptoms the user has
        try:
            num_symptoms = int(user_input)
            current_step = 1
            response = f"Okay, you have {num_symptoms} symptoms. Please enter symptom #1."
        except ValueError:
            response = "Please enter a valid number of symptoms."

    elif current_step <= num_symptoms:
        # Step 2: Collect symptoms one by one
        symptoms.append(user_input)
        if current_step < num_symptoms:
            current_step += 1
            response = f"Please enter symptom #{current_step}."
        else:
            # Step 3: Make the prediction
            response = predict_disease_chatbot()

    # Append the user input and chatbot response to the chat history
    chat_history.append((f"User: {user_input}", f"Chatbot: {response}"))
    
    return chat_history, chat_history  # Return the updated chat history and state

# Gradio chatbot interface with a submit button
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=[gr.Textbox(label="Enter your message", placeholder="Please Enter How many symptoms you have....", lines=1), gr.State([])],  # Textbox input and chat history state
    outputs=[gr.Chatbot(), gr.State([])],  # Output chatbot conversation and chat history state
    title="Disease Prediction Chatbot",
    description="Chatbot to predict possible diseases based on symptoms. First, enter the number of symptoms, and then enter each symptom one by one.",
    allow_flagging="never"
)

# Launch the chatbot interface
iface.launch()


* Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [45]:
import gradio as gr
import pandas as pd
from collections import Counter

# Assume `dis_sym_data_v1`, `doc_data`, `des_data`, `algorithms`, and `le` are preloaded or defined elsewhere

# Column names excluding 'Disease'
test_col = [col for col in dis_sym_data_v1.columns if col != 'Disease']

# Store chatbot state globally
symptoms = []
num_symptoms = 0
current_step = 0

def reset_state():
    global symptoms, num_symptoms, current_step
    symptoms = []
    num_symptoms = 0
    current_step = 0

# Function to predict disease once all symptoms are gathered
def predict_disease_chatbot():
    test_data = {}
    predicted = []

    # Prepare test data for prediction
    for column in test_col:
        test_data[column] = 1 if column in symptoms else 0
    test_df = pd.DataFrame(test_data, index=[0])

    # Predict using multiple algorithms and collect predictions
    for model_name, values in algorithms.items():
        predict_disease = values["model"].predict(test_df)
        predict_disease = le.inverse_transform(predict_disease)
        predicted.extend(predict_disease)

    # Calculate the chances of each predicted disease
    disease_counts = Counter(predicted)
    percentage_per_disease = {disease: (count / len(algorithms)) * 100 for disease, count in disease_counts.items()}

    # Convert to DataFrame
    result_df = pd.DataFrame({"Disease": list(percentage_per_disease.keys()),
                              "Chances": list(percentage_per_disease.values())})

    # Merge additional information (specialist and description)
    result_df = result_df.merge(doc_data, on='Disease', how='left')
    result_df = result_df.merge(des_data, on='Disease', how='left')

    # Format the response
    response = "Here are the predictions based on your symptoms:\n"
    for index, row in result_df.iterrows():
        specialist_info = row.get('Specialist', 'Specialist info not available')
        description_info = row.get('Description', 'Description not available')
        response += f"Disease: {row['Disease']}\nChances: {row['Chances']:.2f}%\nSpecialist: {specialist_info}\nDescription: {description_info}\n\n"

    reset_state()  # Reset state after prediction
    return response

# Function to handle user inputs step by step, while maintaining the chat history
def chatbot_interface(user_input, chat_history=[]):
    global symptoms, num_symptoms, current_step

    if current_step == 0:
        # Step 1: Ask how many symptoms the user has
        try:
            num_symptoms = int(user_input)
            current_step = 1
            response = f"Okay, you have {num_symptoms} symptoms. Please enter symptom #1."
        except ValueError:
            response = "Please enter a valid number of symptoms."

    elif current_step <= num_symptoms:
        # Step 2: Collect symptoms one by one
        symptoms.append(user_input)
        if current_step < num_symptoms:
            current_step += 1
            response = f"Please enter symptom #{current_step}."
        else:
            # Step 3: Make the prediction
            response = predict_disease_chatbot()

    # Append the user input and chatbot response to the chat history
    chat_history.append((f"User: {user_input}", f"Chatbot: {response}"))
    
    return chat_history, chat_history  # Return the updated chat history and state

# Gradio chatbot interface with a submit button
iface = gr.Interface(
    fn=chatbot_interface,
    inputs=[gr.Textbox(label="Enter your message", placeholder="Please Enter How many symptoms you have....", lines=1), gr.State([])],  # Textbox input and chat history state
    outputs=[gr.Chatbot(), gr.State([])],  # Output chatbot conversation and chat history state
    title="Disease Prediction Chatbot",
    description="Chatbot to predict possible diseases based on symptoms. First, enter the number of symptoms, and then enter each symptom one by one.",
    allow_flagging="never"
)

# Launch the chatbot interface
iface.launch()


* Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.
